In [ ]:
import os
import time
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
import jax
import jax.numpy as jnp
import optax
from jax.scipy.spatial.transform import Rotation
from orbax.checkpoint import PyTreeCheckpointer
from flax.training.train_state import TrainState
from matplotlib import pyplot as plt

from lotf import LOTF_PATH
from lotf.envs import HoveringStateEnv, rollout
from lotf.envs.wrappers import EnvWrapper, MinMaxObservationWrapper
from lotf.modules import MLP, LatentConditionedResidualDynamics
from lotf.objects import Quadrotor

%matplotlib inline

# Training Latent-Conditioned Residual Dynamics

## 1. Seed

In [ ]:
seed = 0
key = jax.random.key(seed)
key_init, key_bptt = jax.random.split(key, 2)

## 2. Define Simulation Dynamics Config

In [ ]:
# simulation dynamics config
sim_dyn_config = {
    "use_high_fidelity": True,           # stand-in for the disturbed/real system
    "use_forward_residual": False,       # whether to use residual dynamics in forward simulation
}

# No residual parameters are read when use_forward_residual=False. Passing an
# empty pytree avoids restoring the legacy CUDA-sharded dummy checkpoint.
dummy_residual_params = {}

## 3. Create Quadrotor Object and Data-Collection Environment

In [ ]:
# Apply each newly selected action immediately while retaining the original
# action-history buffer and observation shape expected by the policy. The
# wrapped HoveringStateEnv itself remains configured with delay=0.04.
class ImmediateActionWrapper(EnvWrapper):
    def _step(self, state, action, res_model_params, key):
        action = jnp.clip(
            action, self.action_space.low, self.action_space.high
        )

        # The base environment reads its applied control from the delayed
        # buffer. Fill a temporary buffer with the new action so every
        # integration substep receives that action immediately.
        immediate_state = state.replace(
            last_actions=jnp.broadcast_to(action, state.last_actions.shape)
        )
        transition = self._env._step(
            immediate_state, action, res_model_params, key
        )

        # Preserve the real command history for the checkpoint policy's
        # observation; only the action applied to physics bypasses delay.
        action_history = jnp.roll(state.last_actions, shift=-1, axis=0)
        action_history = action_history.at[-1].set(action)
        next_state = transition.state.replace(last_actions=action_history)
        return transition._replace(
            state=next_state, obs=self._env._get_obs(next_state)
        )

# simulation parameters
sim_dt = 0.02
max_sim_time = 5.0

# quadrotor object
quad_obj = Quadrotor.from_name("example_quad", sim_dyn_config)

eval_env = HoveringStateEnv(
    max_steps_in_episode=int(max_sim_time / sim_dt),
    dt=sim_dt,
    delay=0.04,
    quad_obj=quad_obj,
    margin=0.5,
    hover_target=[1.5, 0.0, 1.5],
    apply_hidden_acceleration=True,  # equation (2): v_dot += w
)
eval_env = ImmediateActionWrapper(eval_env)
eval_env = MinMaxObservationWrapper(eval_env)

# get dims
action_dim = eval_env.action_space.shape[0]
obs_dim = eval_env.observation_space.shape[0]

print("====== eval_env info ======")
print(f"action_dim: {action_dim}")
print(f"obs_dim: {obs_dim}")

## 4. Load the Trained Policy

In [ ]:
policy_name = "state_hovering_params"

# create policy network
base_policy_net = MLP(
    [obs_dim, 512, 512, action_dim],
    action_bias=eval_env.hovering_action,
)

path = LOTF_PATH + "/../checkpoints/policy/" + policy_name
ckptr = PyTreeCheckpointer()
base_policy_params = ckptr.restore(path)
loaded_train_state = TrainState.create(apply_fn=base_policy_net.apply, params=base_policy_params, tx=optax.adam(1e-3))

# define policy function
def policy_trained(obs, key):
    return loaded_train_state.apply_fn(loaded_train_state.params, obs)

## 5. Collect the Transition Dataset

In [ ]:
# The paper stores raw (s_t, a_t, s_{t+1}) transitions. The fixed latent and
# hidden acceleration selected at reset are retained with every transition.
num_rollouts = 256
key, key_rollouts = jax.random.split(key)
rollout_keys = jax.random.split(key_rollouts, num_rollouts)
parallel_rollout = jax.vmap(rollout, in_axes=(None, 0, None, None))
trajectories = parallel_rollout(
    eval_env, rollout_keys, policy_trained, dummy_residual_params
)

env_states = trajectories.state
quad_states = env_states.quadrotor_state

# Convert rotation matrices to the paper's state convention: (p, q, v).
def state_vectors(position, rotation_matrix, velocity):
    quaternion_xyzw = Rotation.from_matrix(rotation_matrix).as_quat()
    quaternion_wxyz = jnp.concatenate(
        [quaternion_xyzw[..., 3:], quaternion_xyzw[..., :3]], axis=-1
    )
    return jnp.concatenate([position, quaternion_wxyz, velocity], axis=-1)

states = state_vectors(
    quad_states.p[:, :-1], quad_states.R[:, :-1], quad_states.v[:, :-1]
)
next_states = state_vectors(
    quad_states.p[:, 1:], quad_states.R[:, 1:], quad_states.v[:, 1:]
)

# The action chosen from s_t is stored at the end of the successor state's
# action buffer, hence the 1: alignment below.
actions = env_states.last_actions[:, 1:, -1, :]
latent_z = env_states.latent_z[:, :-1, :]
hidden_acceleration = env_states.hidden_acceleration[:, :-1, :]
state_actions = jnp.concatenate([states, actions], axis=-1)

# rollout(..., real_step=False) does not auto-reset. Keep the terminal
# transition itself, but discard any subsequent out-of-episode transitions.
done = jnp.logical_or(
    trajectories.terminated[:, 1:], trajectories.truncated[:, 1:]
)
valid_transition = (jnp.cumsum(done, axis=1) - done) == 0

dataset = {
    "state": states[valid_transition],
    "action": actions[valid_transition],
    "state_action": state_actions[valid_transition],
    "next_state": next_states[valid_transition],
    "latent_z": latent_z[valid_transition],
    "hidden_acceleration": hidden_acceleration[valid_transition],
}

assert dataset["state"].shape[-1] == 10
assert dataset["action"].shape[-1] == 4
assert dataset["state_action"].shape[-1] == 14
assert dataset["latent_z"].shape[-1] == 12
assert jnp.all(env_states.latent_z == env_states.latent_z[:, :1, :])

_, samples_per_condition = jnp.unique(
    dataset["hidden_acceleration"], axis=0, return_counts=True
)
print(f"Collected {dataset['state'].shape[0]:,} transitions")
print(f"Conditions represented: {samples_per_condition.shape[0]}/17")
print(f"Samples per represented condition: {samples_per_condition}")

## 6. Instantiate the Latent-Conditioned Residual Model

In [ ]:
# The paper's hovering model predicts position and velocity corrections only.
# It uses a 12-D latent, two 256-unit FiLM blocks, and GELU activations.
residual_model = LatentConditionedResidualDynamics(
    state_action_dim=dataset["state_action"].shape[-1],
    latent_dim=dataset["latent_z"].shape[-1],
    hidden_dim=256,
    num_blocks=2,
    predict_orientation=False,
)

key, key_model = jax.random.split(key)
residual_params = residual_model.initialize(key_model)
residual_train_state = TrainState.create(
    apply_fn=residual_model.apply,
    params=residual_params,
    tx=optax.adam(learning_rate=3e-4),
)

dummy_prediction = residual_model.apply(
    residual_train_state.params,
    dataset["state_action"][0],
    dataset["latent_z"][0],
)
assert dummy_prediction.shape == (6,)
print(f"Residual output shape: {dummy_prediction.shape}")

## 7. Compute Residual Targets and Minimize $L_{dyn}$

In [ ]:
# Build the analytical rigid-body prior used in the paper. Using _step keeps
# its immediate-action handling exactly aligned with collected transitions.
prior_dyn_config = {
    "use_high_fidelity": False,
    "use_forward_residual": False,
}
prior_env = HoveringStateEnv(
    max_steps_in_episode=int(max_sim_time / sim_dt),
    dt=sim_dt,
    delay=0.04,
    quad_obj=Quadrotor.from_name("example_quad", prior_dyn_config),
    apply_hidden_acceleration=False,  # f_prior cannot observe w
    margin=0.5,
    hover_target=[1.5, 0.0, 1.5],
)
prior_env = ImmediateActionWrapper(prior_env)

# Recover the full pre-transition states in the same order as the flattened
# dataset so the prior sees the correct action history and physical state.
transition_states = jax.tree.map(
    lambda leaf: leaf[:, :-1][valid_transition], env_states
)
observed_next_quad_states = jax.tree.map(
    lambda leaf: leaf[:, 1:][valid_transition], quad_states
)
key, key_prior = jax.random.split(key)
prior_keys = jax.random.split(key_prior, dataset["state"].shape[0])

@jax.jit
def evaluate_prior(states, actions, keys):
    def prior_step(state, action, step_key):
        transition = prior_env._step(state, action, {}, step_key)
        return transition.state.quadrotor_state

    return jax.vmap(prior_step)(states, actions, keys)

prior_next_quad_states = evaluate_prior(
    transition_states, dataset["action"], prior_keys
)

def state_difference(observed, nominal, include_orientation=False):
    # Position and velocity live in Euclidean space and use subtraction.
    differences = [observed.p - nominal.p, observed.v - nominal.v]
    if include_orientation:
        # Never subtract quaternions. For R_pred = R_prior Exp(delta_xi),
        # the correct target is Log(R_prior^T R_observed) in R^3.
        relative_rotation = (
            jnp.swapaxes(nominal.R, -1, -2) @ observed.R
        )
        delta_orientation = Rotation.from_matrix(
            relative_rotation
        ).as_rotvec()
        differences.append(delta_orientation)
    return jnp.concatenate(differences, axis=-1)

residual_targets = state_difference(
    observed_next_quad_states,
    prior_next_quad_states,
    include_orientation=residual_model.predict_orientation,
)
assert residual_targets.shape == (dataset["state"].shape[0], 6)

# Split before computing normalization statistics. The paper normalizes
# state-action inputs using statistics from the training buffer.
num_samples = residual_targets.shape[0]
key, key_split = jax.random.split(key)
permutation = jax.random.permutation(key_split, num_samples)
num_train = int(0.9 * num_samples)
train_indices = permutation[:num_train]
validation_indices = permutation[num_train:]

input_mean = dataset["state_action"][train_indices].mean(axis=0)
input_std = dataset["state_action"][train_indices].std(axis=0)
input_std = jnp.maximum(input_std, 1e-6)
normalized_inputs = (dataset["state_action"] - input_mean) / input_std

train_inputs = normalized_inputs[train_indices]
train_latents = dataset["latent_z"][train_indices]
train_targets = residual_targets[train_indices]
validation_inputs = normalized_inputs[validation_indices]
validation_latents = dataset["latent_z"][validation_indices]
validation_targets = residual_targets[validation_indices]

huber_delta = 1.0
batch_size = min(2048, num_train)
num_training_steps = 5000

def dynamics_loss(params, inputs, latents, targets):
    predictions = residual_model.apply(params, inputs, latents)
    # Mean over both transitions and residual coordinates implements L_dyn.
    return optax.huber_loss(
        predictions, targets, delta=huber_delta
    ).mean()

@jax.jit
def train_residual_model(train_state, rng_key):
    def training_step(carry, _):
        state, step_key = carry
        step_key, sample_key = jax.random.split(step_key)
        batch_indices = jax.random.randint(
            sample_key, (batch_size,), 0, train_inputs.shape[0]
        )

        loss, gradients = jax.value_and_grad(dynamics_loss)(
            state.params,
            train_inputs[batch_indices],
            train_latents[batch_indices],
            train_targets[batch_indices],
        )
        state = state.apply_gradients(grads=gradients)
        return (state, step_key), loss

    (train_state, _), losses = jax.lax.scan(
        training_step, (train_state, rng_key), None, num_training_steps
    )
    return train_state, losses

key, key_train = jax.random.split(key)
compiled_train_residual_model = train_residual_model.lower(
    residual_train_state, key_train
).compile()
training_start_time = time.perf_counter()
residual_train_state, training_losses = compiled_train_residual_model(
    residual_train_state, key_train
)
training_losses.block_until_ready()
training_duration_seconds = time.perf_counter() - training_start_time
validation_loss = dynamics_loss(
    residual_train_state.params,
    validation_inputs,
    validation_latents,
    validation_targets,
)

print(f"Initial minibatch L_dyn: {training_losses[0]:.6e}")
print(f"Final minibatch L_dyn:   {training_losses[-1]:.6e}")
print(f"Validation L_dyn:        {validation_loss:.6e}")
print(f"Training time:           {training_duration_seconds:.2f} s")

## 8. Plot $L_{dyn}$ over Training Time

In [ ]:
# lax.scan returns one loss per optimizer update. Distribute the measured
# compiled runtime over those updates to obtain elapsed training time.
elapsed_training_time = jnp.linspace(
    0.0, training_duration_seconds, training_losses.shape[0]
)
loss_for_plot = jnp.maximum(training_losses, 1e-12)

smoothing_window = min(100, training_losses.shape[0])
smoothing_kernel = jnp.ones(smoothing_window) / smoothing_window
smoothed_loss = jnp.convolve(
    loss_for_plot, smoothing_kernel, mode="valid"
)
smoothed_time = elapsed_training_time[smoothing_window - 1:]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(
    elapsed_training_time, loss_for_plot,
    color="tab:blue", alpha=0.2, linewidth=0.8, label="Minibatch $L_{dyn}$"
)
ax.plot(
    smoothed_time, smoothed_loss,
    color="tab:blue", linewidth=2.0,
    label=f"Moving average ({smoothing_window} updates)"
)
ax.axhline(
    validation_loss, color="tab:orange", linestyle="--",
    label=f"Validation $L_{{dyn}}$ = {validation_loss:.2e}"
)
ax.set_xlabel("Elapsed training time [s]")
ax.set_ylabel("Huber dynamics loss $L_{dyn}$")
ax.set_yscale("log")
ax.set_title("Residual Dynamics Training")
ax.grid(True, which="both", alpha=0.25)
ax.legend()
fig.tight_layout()
plt.show()

## 9. Test Five Hidden Wind Conditions at the Same State

In [ ]:
# Use zero and four cardinal unit accelerations. Each row retrieves the
# immutable latent associated with that condition; w itself is not a model input.
test_condition_indices = jnp.array([0, 1, 3, 5, 7])
test_pairs = eval_env.wind_z_table[test_condition_indices]
test_hidden_accelerations = test_pairs[:, :3]
test_latents = test_pairs[:, 3:]

# Hold s_t and a_t exactly fixed across all five tests.
fixed_state_action = dataset["state_action"][validation_indices[0]]
fixed_normalized_input = (fixed_state_action - input_mean) / input_std
fixed_inputs = jnp.broadcast_to(
    fixed_normalized_input, (test_condition_indices.shape[0], 14)
)

test_residual_predictions = residual_model.apply(
    residual_train_state.params, fixed_inputs, test_latents
)

# Five explicit tests: every condition-associated prediction must differ from
# all four predictions produced for the other hidden conditions.
prediction_tolerance = 1e-6
for test_idx in range(test_condition_indices.shape[0]):
    other_predictions = jnp.concatenate(
        [
            test_residual_predictions[:test_idx],
            test_residual_predictions[test_idx + 1:],
        ],
        axis=0,
    )
    distances = jnp.linalg.norm(
        other_predictions - test_residual_predictions[test_idx], axis=-1
    )
    assert jnp.all(distances > prediction_tolerance), (
        f"Condition {test_idx} produced a collapsed residual prediction. "
        "The training data may not contain condition-dependent dynamics."
    )

assert jnp.unique(test_hidden_accelerations, axis=0).shape[0] == 5
assert jnp.unique(test_latents, axis=0).shape[0] == 5

print(f"Fixed state-action: {fixed_state_action}")
for acceleration, prediction in zip(
    test_hidden_accelerations, test_residual_predictions
):
    print(
        f"hidden acceleration={acceleration} "
        f"-> predicted [delta_p, delta_v]={prediction}"
    )
print("All five hidden-condition tests passed.")

## 10. Evaluate Predicted Residuals on Fresh Policy Rollouts

In [ ]:
# Generate a held-out dataset with RNG keys that were never used for training.
num_test_rollouts = 64
key, key_test_rollouts = jax.random.split(key)
test_rollout_keys = jax.random.split(key_test_rollouts, num_test_rollouts)
test_trajectories = parallel_rollout(
    eval_env, test_rollout_keys, policy_trained, dummy_residual_params
)

test_env_states = test_trajectories.state
test_quad_states = test_env_states.quadrotor_state
test_states = state_vectors(
    test_quad_states.p[:, :-1],
    test_quad_states.R[:, :-1],
    test_quad_states.v[:, :-1],
)
test_actions = test_env_states.last_actions[:, 1:, -1, :]
test_state_actions = jnp.concatenate([test_states, test_actions], axis=-1)
test_done = jnp.logical_or(
    test_trajectories.terminated[:, 1:],
    test_trajectories.truncated[:, 1:],
)
test_valid_transition = (jnp.cumsum(test_done, axis=1) - test_done) == 0

# Evaluate the analytical prior from exactly the same pre-transition states.
test_transition_states = jax.tree.map(
    lambda leaf: leaf[:, :-1][test_valid_transition], test_env_states
)
test_observed_next_quad_states = jax.tree.map(
    lambda leaf: leaf[:, 1:][test_valid_transition], test_quad_states
)
test_actions = test_actions[test_valid_transition]
test_state_actions = test_state_actions[test_valid_transition]
test_latent_z = test_env_states.latent_z[:, :-1][test_valid_transition]

key, key_test_prior = jax.random.split(key)
test_prior_keys = jax.random.split(
    key_test_prior, test_state_actions.shape[0]
)
test_prior_next_quad_states = evaluate_prior(
    test_transition_states, test_actions, test_prior_keys
)

# Ground-truth held-out residual: s_next minus the analytical prior. For
# hovering this is Euclidean position and velocity difference.
test_residual_targets = state_difference(
    test_observed_next_quad_states,
    test_prior_next_quad_states,
    include_orientation=residual_model.predict_orientation,
)
test_normalized_inputs = (test_state_actions - input_mean) / input_std
test_residual_predictions = residual_model.apply(
    residual_train_state.params, test_normalized_inputs, test_latent_z
)

# Quantitative comparison against both ground truth and a zero-residual model.
test_errors = test_residual_predictions - test_residual_targets
test_huber_loss = optax.huber_loss(
    test_residual_predictions, test_residual_targets, delta=huber_delta
).mean()
zero_model_huber_loss = optax.huber_loss(
    jnp.zeros_like(test_residual_targets),
    test_residual_targets,
    delta=huber_delta,
).mean()
component_mae = jnp.mean(jnp.abs(test_errors), axis=0)
component_rmse = jnp.sqrt(jnp.mean(test_errors**2, axis=0))
sum_squared_error = jnp.sum(test_errors**2, axis=0)
sum_squared_total = jnp.sum(
    (test_residual_targets - test_residual_targets.mean(axis=0)) ** 2,
    axis=0,
)
component_r2 = 1.0 - sum_squared_error / jnp.maximum(
    sum_squared_total, 1e-12
)

assert test_residual_predictions.shape == test_residual_targets.shape
assert jnp.all(jnp.isfinite(test_residual_predictions))
assert test_huber_loss < zero_model_huber_loss, (
    "The learned residual model does not outperform the analytical prior "
    "on fresh rollouts."
)

component_names = ["delta_px", "delta_py", "delta_pz",
                   "delta_vx", "delta_vy", "delta_vz"]
print(f"Held-out transitions: {test_residual_targets.shape[0]:,}")
print(f"Learned-model Huber L_dyn: {test_huber_loss:.6e}")
print(f"Zero-residual Huber loss:   {zero_model_huber_loss:.6e}")
for name, mae, rmse, r2 in zip(
    component_names, component_mae, component_rmse, component_r2
):
    print(f"{name:>8s}: MAE={mae:.3e}, RMSE={rmse:.3e}, R2={r2:.3f}")

# Predicted-versus-measured residual plots. Position is displayed in mm so
# its O(dt^2) one-step residual is visible. Measured and predicted axes use
# their own data limits so a larger prediction range cannot flatten targets.
plot_stride = max(1, test_residual_targets.shape[0] // 5000)
fig, axes = plt.subplots(2, 3, figsize=(12, 7))

def padded_data_limits(values):
    lower = float(values.min())
    upper = float(values.max())
    span = max(upper - lower, 1e-3 * max(abs(lower), abs(upper)), 1e-12)
    padding = 0.05 * span
    return lower - padding, upper + padding

for component_idx, (axis, component_name) in enumerate(
    zip(axes.flat, component_names)
):
    display_scale = 1000.0 if component_idx < 3 else 1.0
    display_unit = "mm" if component_idx < 3 else "m/s"
    target_component = (
        test_residual_targets[::plot_stride, component_idx] * display_scale
    )
    predicted_component = (
        test_residual_predictions[::plot_stride, component_idx] * display_scale
    )
    measured_limits = padded_data_limits(target_component)
    predicted_limits = padded_data_limits(predicted_component)
    axis.scatter(
        target_component, predicted_component, s=5, alpha=0.25
    )
    identity_lower = max(measured_limits[0], predicted_limits[0])
    identity_upper = min(measured_limits[1], predicted_limits[1])
    if identity_lower < identity_upper:
        axis.plot(
            [identity_lower, identity_upper],
            [identity_lower, identity_upper],
            "k--", linewidth=1, label="Ideal"
        )
    axis.set_xlim(*measured_limits)
    axis.set_ylim(*predicted_limits)
    axis.set_title(f"{component_name} (R2={component_r2[component_idx]:.3f})")
    axis.set_xlabel(f"Measured residual [{display_unit}]")
    axis.set_ylabel(f"Predicted residual [{display_unit}]")
    axis.grid(True, alpha=0.2)

fig.suptitle("Held-Out Residual Dynamics: Prediction vs. Rollout Target")
fig.tight_layout()
plt.show()

## 11. Evaluate Multi-Step Residual-Dynamics Rollouts

In [ ]:
# Reuse the fresh held-out trajectories from section 10. Starting from each
# true initial state, recursively advance f_prior through the recorded action
# sequence and add the learned residual after every analytical transition.
rollout_horizon = test_env_states.time.shape[1] - 1
rollout_actions = test_env_states.last_actions[:, 1:, -1, :]
actions_by_time = jnp.swapaxes(rollout_actions, 0, 1)
initial_rollout_states = jax.tree.map(lambda leaf: leaf[:, 0], test_env_states)
rollout_latents = initial_rollout_states.latent_z
key, key_model_rollouts = jax.random.split(key)
model_rollout_keys = jax.random.split(
    key_model_rollouts, (rollout_horizon, num_test_rollouts)
)

@jax.jit
def predict_multi_step_rollout(
    initial_states, actions, latents, step_keys,
    position_residual_scale, velocity_residual_scale
):
    def rollout_step(predicted_states, step_inputs):
        step_actions, keys_at_step = step_inputs

        # Advance the nominal model with the same immediate-action wrapper.
        prior_transitions = jax.vmap(
            prior_env._step, in_axes=(0, 0, None, 0)
        )(predicted_states, step_actions, {}, keys_at_step)
        prior_states = prior_transitions.state

        # D_psi is evaluated on the recursively predicted state, not on the
        # ground-truth state (no teacher forcing).
        predicted_quad_states = predicted_states.quadrotor_state
        predicted_state_vectors = state_vectors(
            predicted_quad_states.p,
            predicted_quad_states.R,
            predicted_quad_states.v,
        )
        predicted_state_actions = jnp.concatenate(
            [predicted_state_vectors, step_actions], axis=-1
        )
        normalized_state_actions = (
            predicted_state_actions - input_mean
        ) / input_std
        raw_residual_correction = residual_model.apply(
            residual_train_state.params, normalized_state_actions, latents
        )
        position_correction = (
            position_residual_scale * raw_residual_correction[:, :3]
        )
        velocity_correction = (
            velocity_residual_scale * raw_residual_correction[:, 3:6]
        )

        prior_quad_states = prior_states.quadrotor_state
        corrected_quad_states = prior_quad_states.replace(
            p=prior_quad_states.p + position_correction,
            v=prior_quad_states.v + velocity_correction,
        )
        corrected_states = prior_states.replace(
            quadrotor_state=corrected_quad_states
        )
        predicted_output = (
            corrected_quad_states.p,
            corrected_quad_states.v,
            velocity_correction,
            normalized_state_actions,
        )
        return corrected_states, predicted_output

    _, rollout_outputs = jax.lax.scan(
        rollout_step, initial_states, (actions, step_keys)
    )
    return rollout_outputs

# Unit scales evaluate the complete learned residual; zero scales give an
# identical analytical-prior baseline. Scan outputs are time-major.
(predicted_positions, predicted_velocities,
 learned_velocity_corrections, normalized_rollout_inputs) = (
    predict_multi_step_rollout(
        initial_rollout_states,
        actions_by_time,
        rollout_latents,
        model_rollout_keys,
        1.0,
        1.0,
    )
)
(prior_positions, prior_velocities, _, _) = predict_multi_step_rollout(
    initial_rollout_states,
    actions_by_time,
    rollout_latents,
    model_rollout_keys,
    0.0,
    0.0,
)

# Convert predictions to [environment, time, xyz] and compare only valid
# pre-termination portions of the held-out trajectories.
predicted_positions = jnp.swapaxes(predicted_positions, 0, 1)
predicted_velocities = jnp.swapaxes(predicted_velocities, 0, 1)
learned_velocity_corrections = jnp.swapaxes(
    learned_velocity_corrections, 0, 1
)
normalized_rollout_inputs = jnp.swapaxes(
    normalized_rollout_inputs, 0, 1
)
prior_positions = jnp.swapaxes(prior_positions, 0, 1)
prior_velocities = jnp.swapaxes(prior_velocities, 0, 1)
true_positions = test_quad_states.p[:, 1:]
true_velocities = test_quad_states.v[:, 1:]
multi_step_valid = test_valid_transition

residual_position_error = jnp.linalg.norm(
    predicted_positions - true_positions, axis=-1
)
prior_position_error = jnp.linalg.norm(
    prior_positions - true_positions, axis=-1
)
residual_velocity_error = jnp.linalg.norm(
    predicted_velocities - true_velocities, axis=-1
)
prior_velocity_error = jnp.linalg.norm(
    prior_velocities - true_velocities, axis=-1
)

def masked_time_mean(values, valid):
    valid_count = jnp.maximum(valid.sum(axis=0), 1)
    return jnp.where(valid, values, 0.0).sum(axis=0) / valid_count

def masked_time_max(values, valid):
    maxima = jnp.where(valid, values, -jnp.inf).max(axis=0)
    return jnp.where(valid.any(axis=0), maxima, jnp.nan)

def masked_rmse(vector_error, valid):
    squared_norm = jnp.sum(vector_error**2, axis=-1)
    return jnp.sqrt(
        jnp.where(valid, squared_norm, 0.0).sum()
        / jnp.maximum(valid.sum(), 1)
    )

residual_position_curve = masked_time_mean(
    residual_position_error, multi_step_valid
)
prior_position_curve = masked_time_mean(
    prior_position_error, multi_step_valid
)
residual_velocity_curve = masked_time_mean(
    residual_velocity_error, multi_step_valid
)
prior_velocity_curve = masked_time_mean(
    prior_velocity_error, multi_step_valid
)

residual_position_rmse = masked_rmse(
    predicted_positions - true_positions, multi_step_valid
)
prior_position_rmse = masked_rmse(
    prior_positions - true_positions, multi_step_valid
)
residual_velocity_rmse = masked_rmse(
    predicted_velocities - true_velocities, multi_step_valid
)
prior_velocity_rmse = masked_rmse(
    prior_velocities - true_velocities, multi_step_valid
)

print("Overall open-loop multi-step RMSE")
print(
    f"  Position: residual={residual_position_rmse:.4f} m, "
    f"prior={prior_position_rmse:.4f} m"
)
print(
    f"  Velocity: residual={residual_velocity_rmse:.4f} m/s, "
    f"prior={prior_velocity_rmse:.4f} m/s"
)
for horizon_seconds in (0.5, 1.0, 2.0, 5.0):
    horizon_idx = min(
        int(round(horizon_seconds / sim_dt)) - 1, rollout_horizon - 1
    )
    print(
        f"t={horizon_seconds:>3.1f}s | "
        f"position residual/prior: "
        f"{residual_position_curve[horizon_idx]:.4f}/"
        f"{prior_position_curve[horizon_idx]:.4f} m | "
        f"velocity residual/prior: "
        f"{residual_velocity_curve[horizon_idx]:.4f}/"
        f"{prior_velocity_curve[horizon_idx]:.4f} m/s"
    )

rollout_time = sim_dt * jnp.arange(1, rollout_horizon + 1)
network_input_time = sim_dt * jnp.arange(rollout_horizon)
fig, (position_axis, velocity_axis) = plt.subplots(1, 2, figsize=(12, 4.5))
position_axis.plot(
    rollout_time, prior_position_curve, label="Analytical prior",
    color="tab:orange", linestyle="--"
)
position_axis.plot(
    rollout_time, residual_position_curve, label="Prior + learned residual",
    color="tab:blue"
)
position_axis.set_xlabel("Rollout time [s]")
position_axis.set_ylabel("Mean position error [m]")
position_axis.set_title("Open-Loop Position Prediction")
position_axis.grid(True, alpha=0.25)
position_axis.legend()

velocity_axis.plot(
    rollout_time, prior_velocity_curve, label="Analytical prior",
    color="tab:orange", linestyle="--"
)
velocity_axis.plot(
    rollout_time, residual_velocity_curve, label="Prior + learned residual",
    color="tab:blue"
)
velocity_axis.set_xlabel("Rollout time [s]")
velocity_axis.set_ylabel("Mean velocity error [m/s]")
velocity_axis.set_title("Open-Loop Velocity Prediction")
velocity_axis.grid(True, alpha=0.25)
velocity_axis.legend()

fig.suptitle("Multi-Step Residual-Dynamics Evaluation")
fig.tight_layout()
plt.show()

# normalized_rollout_inputs contains the exact 14-D state-action input
# evaluated by D_psi for every recursively predicted rollout state. At each
# time, retain the valid state whose normalized network-input norm is largest.
normalized_input_norms = jnp.linalg.norm(
    normalized_rollout_inputs, axis=-1
)
max_normalized_input_curve = masked_time_max(
    normalized_input_norms, multi_step_valid
)

# Apply the same training normalization to every true held-out pre-transition
# state and recorded action, preserving [environment, time, feature] layout.
true_state_vectors = state_vectors(
    test_quad_states.p[:, :-1],
    test_quad_states.R[:, :-1],
    test_quad_states.v[:, :-1],
)
true_state_actions = jnp.concatenate(
    [true_state_vectors, rollout_actions], axis=-1
)
normalized_true_inputs = (true_state_actions - input_mean) / input_std
normalized_true_input_norms = jnp.linalg.norm(
    normalized_true_inputs, axis=-1
)
max_normalized_true_input_curve = masked_time_max(
    normalized_true_input_norms, multi_step_valid
)

# This is the physical correction actually added to v after every prior
# transition (velocity_residual_scale=1), rather than its normalized value.
learned_velocity_correction_magnitudes = jnp.linalg.norm(
    learned_velocity_corrections, axis=-1
)
mean_velocity_correction_curve = masked_time_mean(
    learned_velocity_correction_magnitudes, multi_step_valid
)
max_velocity_correction_curve = masked_time_max(
    learned_velocity_correction_magnitudes, multi_step_valid
)

fig, normalized_axis = plt.subplots(figsize=(8, 4.5))
normalized_axis.plot(
    network_input_time, max_normalized_input_curve, color="tab:purple"
)
normalized_axis.set_xlabel("Rollout time [s]")
normalized_axis.set_ylabel(r"$\max_i \|\widetilde{[s,a]}_{t,i}\|_2$")
normalized_axis.set_title("Largest Normalized Network Input Across States")
normalized_axis.grid(True, alpha=0.25)
fig.tight_layout()
plt.show()

fig, true_normalized_axis = plt.subplots(figsize=(8, 4.5))
true_normalized_axis.plot(
    network_input_time, max_normalized_true_input_curve, color="tab:cyan"
)
true_normalized_axis.set_xlabel("Trajectory time [s]")
true_normalized_axis.set_ylabel(
    r"$\max_i \|\widetilde{[s,a]}^{true}_{t,i}\|_2$"
)
true_normalized_axis.set_title(
    "Largest Normalized Network Input on True Held-Out Trajectories"
)
true_normalized_axis.grid(True, alpha=0.25)
fig.tight_layout()
plt.show()

fig, correction_axis = plt.subplots(figsize=(8, 4.5))
correction_axis.plot(
    rollout_time, mean_velocity_correction_curve,
    color="tab:green", label="Mean across valid states"
)
correction_axis.plot(
    rollout_time, max_velocity_correction_curve,
    color="tab:red", linestyle="--", label="Maximum valid state"
)
correction_axis.set_xlabel("Rollout time [s]")
correction_axis.set_ylabel(r"$\|\Delta v_{learned}\|_2$ [m/s]")
correction_axis.set_title("Actual Learned Velocity Correction")
correction_axis.grid(True, alpha=0.25)
correction_axis.legend()
fig.tight_layout()
plt.show()

print(
    "Peak normalized network-input norm: "
    f"{jnp.nanmax(max_normalized_input_curve):.3f}"
)
print(
    "Peak true-trajectory normalized network-input norm: "
    f"{jnp.nanmax(max_normalized_true_input_curve):.3f}"
)
print(
    "Peak applied velocity correction: "
    f"{jnp.nanmax(max_velocity_correction_curve):.6f} m/s"
)

## 12. Multi-Step Rollouts with the Predicted Position Residual Disabled

In [ ]:
# Keep the learned velocity correction but explicitly force delta_p_pred=0.
# Position can still improve indirectly at later steps because the corrected
# velocity is propagated through p_dot=v by the analytical prior.
(velocity_only_positions, velocity_only_velocities, _, _) = (
    predict_multi_step_rollout(
        initial_rollout_states,
        actions_by_time,
        rollout_latents,
        model_rollout_keys,
        0.0,  # disable the predicted position residual
        1.0,  # retain the predicted velocity residual
    )
)
velocity_only_positions = jnp.swapaxes(velocity_only_positions, 0, 1)
velocity_only_velocities = jnp.swapaxes(velocity_only_velocities, 0, 1)

# At the first prediction step, position must exactly match the prior because
# no learned position correction has yet been applied.
assert jnp.allclose(
    velocity_only_positions[:, 0], prior_positions[:, 0], atol=1e-7
)
assert jnp.all(jnp.isfinite(velocity_only_positions))
assert jnp.all(jnp.isfinite(velocity_only_velocities))

velocity_only_position_error = jnp.linalg.norm(
    velocity_only_positions - true_positions, axis=-1
)
velocity_only_velocity_error = jnp.linalg.norm(
    velocity_only_velocities - true_velocities, axis=-1
)
velocity_only_position_curve = masked_time_mean(
    velocity_only_position_error, multi_step_valid
)
velocity_only_velocity_curve = masked_time_mean(
    velocity_only_velocity_error, multi_step_valid
)
velocity_only_position_rmse = masked_rmse(
    velocity_only_positions - true_positions, multi_step_valid
)
velocity_only_velocity_rmse = masked_rmse(
    velocity_only_velocities - true_velocities, multi_step_valid
)

print("Overall open-loop multi-step RMSE")
print(
    f"  Position | prior={prior_position_rmse:.4f} m, "
    f"full residual={residual_position_rmse:.4f} m, "
    f"velocity-only residual={velocity_only_position_rmse:.4f} m"
)
print(
    f"  Velocity | prior={prior_velocity_rmse:.4f} m/s, "
    f"full residual={residual_velocity_rmse:.4f} m/s, "
    f"velocity-only residual={velocity_only_velocity_rmse:.4f} m/s"
)
for horizon_seconds in (0.5, 1.0, 2.0, 5.0):
    horizon_idx = min(
        int(round(horizon_seconds / sim_dt)) - 1, rollout_horizon - 1
    )
    print(
        f"t={horizon_seconds:>3.1f}s | "
        f"position velocity-only/full/prior: "
        f"{velocity_only_position_curve[horizon_idx]:.4f}/"
        f"{residual_position_curve[horizon_idx]:.4f}/"
        f"{prior_position_curve[horizon_idx]:.4f} m | "
        f"velocity velocity-only/full/prior: "
        f"{velocity_only_velocity_curve[horizon_idx]:.4f}/"
        f"{residual_velocity_curve[horizon_idx]:.4f}/"
        f"{prior_velocity_curve[horizon_idx]:.4f} m/s"
    )

fig, (position_axis, velocity_axis) = plt.subplots(1, 2, figsize=(12, 4.5))
for axis, prior_curve, full_curve, velocity_only_curve, ylabel, title in (
    (
        position_axis, prior_position_curve, residual_position_curve,
        velocity_only_position_curve, "Mean position error [m]",
        "Open-Loop Position Prediction"
    ),
    (
        velocity_axis, prior_velocity_curve, residual_velocity_curve,
        velocity_only_velocity_curve, "Mean velocity error [m/s]",
        "Open-Loop Velocity Prediction"
    ),
):
    axis.plot(
        rollout_time, prior_curve, color="tab:orange", linestyle="--",
        label="Analytical prior"
    )
    # axis.plot(
    #     rollout_time, full_curve, color="tab:blue",
    #     label="Full residual"
    # )
    axis.plot(
        rollout_time, velocity_only_curve, color="tab:green",
        label="Velocity residual only"
    )
    axis.set_xlabel("Rollout time [s]")
    axis.set_ylabel(ylabel)
    axis.set_title(title)
    axis.grid(True, alpha=0.25)
    axis.legend()

fig.suptitle("Effect of Disabling the Predicted Position Residual")
fig.tight_layout()
plt.show()